In [0]:
%run ./_utils

### Export `deleted_ids.csv` (oxjob #784)

Writes the cumulative deleted-works ledger (`openalex.works.deleted_works`,
maintained nightly by `notebooks/end2end/TrackDeletedWorks`) to
`full/{date}/deleted_ids.csv` — one row per deleted work: `work_id`
(`https://openalex.org/W…`, same id form as the works entity files) and
`deleted_date` (the date the work disappeared from `openalex_works`; for works
ledgered by the one-off full-index reconcile this is the detection date, not
the true deletion date). Works that reappear are removed from the ledger, so a
live work never shows up here. Consumers apply the file as: remove these ids
from any locally-held copy of works.


In [0]:
LEDGER = "openalex.works.deleted_works"

date_str = get_snapshot_date()
out_path = f"{S3_BASE}/{date_str}/deleted_ids.csv"
tmp_dir = f"{S3_BASE}/{date_str}/_temp/deleted_csv"

if spark.catalog.tableExists(LEDGER):
    df = (
        spark.table(LEDGER)
        .selectExpr(f"CONCAT('https://openalex.org/W', work_id) AS work_id", "deleted_date")
        .distinct()
        .coalesce(1)
        .sortWithinPartitions("deleted_date", "work_id")
    )
    df.write.mode("overwrite").option("header", True).csv(tmp_dir)
    part = [f.path for f in dbutils.fs.ls(tmp_dir) if f.name.startswith("part-") and f.name.endswith(".csv")][0]
    dbutils.fs.cp(part, out_path)
    dbutils.fs.rm(tmp_dir, recurse=True)
    count = spark.table(LEDGER).select("work_id").distinct().count()
    print(f"Wrote {count:,} deleted works to {out_path}")
else:
    dbutils.fs.put(out_path, "work_id,deleted_date\n", overwrite=True)
    print(f"{LEDGER} does not exist yet; wrote header-only {out_path}")
